# v10.2.30 campaign monitor
Strictly read-only: this notebook only reads campaign artifacts and never controls simulation processes.

In [ ]:
from pathlib import Path
REPOSITORY_ROOT = Path(
    "/Volumes/Data/Data/Nanopillar_calculation/"
    "PF-fracture-fatigue_codex_v10_2_30"
)
CAMPAIGN_ROOT = REPOSITORY_ROOT / "runs" / "CAMPAIGN_NAME"
REFRESH_SECONDS = 30


In [ ]:
import sys, time
sys.path.insert(0, str(REPOSITORY_ROOT))
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from scripts.v10230_jupyter_monitor import campaign_snapshot, recent_event_details, tail, validate_campaign_path
validate_campaign_path(REPOSITORY_ROOT, CAMPAIGN_ROOT)


In [ ]:
output = widgets.Output()
refresh_button = widgets.Button(description="Refresh", icon="refresh")
auto = widgets.ToggleButton(description="Auto refresh", value=False)
case_selector = widgets.Dropdown(description="Case")
def render(_=None):
    snap = campaign_snapshot(REPOSITORY_ROOT, CAMPAIGN_ROOT)
    with output:
        clear_output(wait=True)
        counts=snap['counts']; owner=snap['owner']
        display({"total_cases":len(snap['cases']),"queued":counts.get('pending',0),"active":counts.get('running',0),"completed_growth":counts.get('completed_growth',0),"right_censored_no_growth":counts.get('right_censored_no_growth',0),"right_censored_after_growth":counts.get('right_censored_after_growth',0),"failed":counts.get('failed',0),"incomplete_restartable":counts.get('incomplete_restartable',0),"blocked":counts.get('blocked',0),"supervisor_pid":owner.get('pid'),"supervisor_alive":snap['supervisor_alive'],"maximum_concurrency":snap['matrix'].get('maximum_concurrency',2),"campaign_disk_GiB":snap['disk_bytes']/1024**3,"free_Data_GiB":snap['free_bytes']/1024**3,"launch_HEAD":snap['matrix'].get('launch_git_head'),"simulation_baseline":snap['matrix'].get('qualified_simulation_head'),"family_hash":snap['matrix'].get('family',{}).get('observed_sha256')})
        display(snap['cases'])
        if snap['warnings']: display({"warnings":snap['warnings']})
        frame=snap['cases']
        if not frame.empty:
            fig,axes=plt.subplots(2,3,figsize=(15,8))
            styles={'completed_growth':'o','running':'^','right_censored_no_growth':'x','right_censored_after_growth':'X','failed':'s'}
            for status,group in frame.groupby('status'):
                marker=styles.get(status,'.')
                axes[0,0].scatter(group.cycles,group.extension_um,label=status,marker=marker)
                axes[0,1].scatter(group.cycles,group.event_count,label=status,marker=marker)
                active_group=group[group.status=='running']; axes[0,2].scatter(active_group['fraction'],active_group.action_fraction,label=status,marker=marker)
                rates=group[group.developed_da_dN.notna()]
                axes[1,0].scatter(rates.deltaK,rates.developed_da_dN,label=status,marker=marker)
                axes[1,1].scatter(rates.fraction,rates.developed_da_dN,label=status,marker=marker)
                axes[1,2].scatter(group.fraction,group.output_size_bytes/1024**2,label=status,marker=marker)
            axes[0,0].set(xscale='log',xlabel='Cycles',ylabel='Projected extension (um)'); axes[0,1].set(xscale='log',xlabel='Cycles',ylabel='Committed events'); axes[0,2].set(xlabel='Fraction',ylabel='H/Xi'); axes[1,0].set(yscale='log',xlabel='DeltaK',ylabel='Developed da/dN'); axes[1,1].set(yscale='log',xlabel='Fraction',ylabel='Developed da/dN'); axes[1,2].set(xlabel='Fraction',ylabel='Output size (MiB)')
            axes[0,0].legend(fontsize=7); fig.tight_layout(); plt.show()
        names=list(frame.case) if not frame.empty else []; case_selector.options=names
refresh_button.on_click(render)
display(widgets.HBox([refresh_button,auto,case_selector]), output)
render()


In [ ]:
detail = widgets.Output()
def show_case(change=None):
    if not case_selector.value: return
    case=CAMPAIGN_ROOT/case_selector.value; out=case/'output'
    with detail:
        clear_output(wait=True)
        display(recent_event_details(case,20))
        print(tail(case/'run.log',80) or tail(out/'run.log',80))
case_selector.observe(show_case,names='value'); display(detail); show_case()


In [ ]:
# Run this cell for optional auto-refresh; toggle off or interrupt the kernel to stop cleanly.
while auto.value:
    render(); time.sleep(max(1, REFRESH_SECONDS))
